# Three sampling planners for a unicycle

A unicycle can move forward and turn, yet it cannot translate sideways. This example compares MPPI, CEM, and predictive sampling on the same terminal pose and control-effort graph. The model, two-second horizon, goal and cost scales match the [C++ example](../../../../gtsam_unstable/examples/mppi/UnicycleSamplingExample.cpp). The notebook uses a smaller batch and fewer iterations for the Python dynamics callback.

GTSAM Copyright 2010–2026, Georgia Tech Research Corporation, Atlanta, Georgia. See LICENSE. Authors: Frank Dellaert and Akash Ratheesh.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/borglab/gtsam/blob/develop/python/gtsam_unstable/examples/planning/UnicycleSamplingExample.ipynb)

In [1]:
import sys
if 'google.colab' in sys.modules:
    %pip install --quiet gtsam-develop plotly

In [2]:
import numpy as np
import plotly.graph_objects as go
import gtsam
from gtsam.symbol_shorthand import U, X
from IPython.display import HTML, display

## Build the cost graph

The prior on the last pose penalizes position and heading error in local coordinates. A Gaussian prior on each control penalizes high speed and yaw rate. The first pose and nominal controls provide the starting point for shooting.

In [3]:
horizon, dt = 20, 0.1
states = [X(k) for k in range(horizon + 1)]
actions = [U(k) for k in range(horizon)]
goal = gtsam.Pose2(1.5, 0.75, 0.8)
graph, initial = gtsam.NonlinearFactorGraph(), gtsam.Values()
initial.insert(states[0], gtsam.Pose2())
effort = gtsam.noiseModel.Isotropic.Sigma(2, 4.0)
terminal = gtsam.noiseModel.Diagonal.Sigmas(np.array([0.1, 0.1, 0.2]))
for key in actions:
    initial.insert(key, np.zeros(2))
    graph.add(gtsam.PriorFactorVector(key, np.zeros(2), effort))
graph.add(gtsam.PriorFactorPose2(states[-1], goal, terminal))

## Integrate and compare

The dynamics return the body-frame velocity $(v,0,\omega)$. The planner multiplies it by the time step and retracts it at the current pose. Each search gets the same initial plan and random seed. MPPI averages candidates with exponential weights, CEM fits the proposal to its elites, and predictive sampling keeps the cheapest trajectory including the incumbent.

In [4]:
def dynamics(pose, u):
    return np.array([u[0], 0.0, u[1]])

planner = gtsam.SamplingPlannerPose2Vector(states, actions, dt)
baseline = planner.rollout(initial, np.zeros((2, horizon)), dynamics)
initial_cost = graph.error(baseline)
algorithms = {"MPPI": gtsam.MPPI(), "CEM": gtsam.CEM(8),
              "Predictive sampling": gtsam.PredictiveSampling()}
plans = {}
for name, algorithm in algorithms.items():
    proposal = gtsam.GaussianProposal(np.eye(2), 64, 42)
    plan = planner.optimize(graph, initial, dynamics, proposal, algorithm, 8)
    plans[name] = plan
    print(f"{name:20s} graph cost {graph.error(plan):7.3f} (initial {initial_cost:.3f})")
    assert graph.error(plan) < initial_cost

MPPI                 graph cost   3.198 (initial 148.625)


CEM                  graph cost   1.123 (initial 148.625)
Predictive sampling  graph cost   3.237 (initial 148.625)


The curves show where each optimized control sequence drives the robot. Orientation contributes to the reported graph cost. Interactive charts require an internet connection.

In [5]:
figure = go.Figure()
for name, plan in plans.items():
    poses = [plan.atPose2(key) for key in states]
    figure.add_scatter(x=[p.x() for p in poses], y=[p.y() for p in poses],
                       mode="lines+markers", name=name)
figure.add_scatter(x=[goal.x()], y=[goal.y()], mode="markers", name="Goal",
                   marker=dict(symbol="x", size=12))
figure.update_layout(xaxis_title="x (m)", yaxis_title="y (m)")
figure.update_yaxes(scaleanchor="x", scaleratio=1)
from html import escape

document = figure.to_html(include_plotlyjs="cdn", config={"displaylogo": False, "responsive": True})
display(HTML(f'<iframe title="Three sampling planners for a unicycle" style="width:100%;border:0" '
             f'height="{figure.layout.height or 480}" srcdoc="{escape(document, quote=True)}"></iframe>'))

/Users/dellaert/miniforge3/envs/py312/lib/python3.12/site-packages/IPython/core/display.py:431: UserWarning:

Consider using IPython.display.IFrame instead

